In [118]:
import pandas as pd
from geopy.distance import great_circle
df = pd.read_csv("../data/raw/fraudTrain.csv")
pd.set_option('display.max_columns', None)

In [119]:
def clean_data(df:pd.DataFrame):
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
    df["date_of_birth"] = pd.to_datetime(df["dob"])
    
    df.drop(columns=["Unnamed: 0","dob"],inplace=True)
    
    
def add_time(df:pd.DataFrame):
    df["week_day"] = df["trans_date_trans_time"].dt.day_name()
    df["hour"] = df["trans_date_trans_time"].dt.hour
    
def add_age(df:pd.DataFrame):
    df["age"] = df["trans_date_trans_time"].dt.year - df["date_of_birth"].dt.year
    bins = [0, 17, 24, 34, 49, 64, 100]
    labels = ["0-17", "18-24", "25-34","35-49", "50-64","65+"]
    df['age_group'] = pd.cut(df['age'],right=True, bins=bins, labels=labels, include_lowest=True)

def add_city_group(df:pd.DataFrame):
    bins = [0, 1000, 10000, 100000,500000, 1000000, 10000000]
    labels = ["rural","small_town","small_city","medium_city","large_city","mega_city"]
    df['city_group'] = pd.cut(df['city_pop'],right=True, bins=bins, labels=labels, include_lowest=True)

def calc_distance(user_lat:float, user_long:float, merch_lat:float, merch_long:float):
    point_user = (user_lat, user_long)
    point_merch = (merch_lat, merch_long)
    return great_circle(point_user, point_merch).km
def add_distance(df:pd.DataFrame):
    df["distance"] = df.apply(lambda row: calc_distance(row["lat"], row["long"], row["merch_lat"], row["merch_long"]), axis=1)

def add_distance_bucket(df:pd.DataFrame):
    bins = [0, 1, 10, 20, 100, 1000, 10000]
    labels = ["district","city","city_suburbs","metropolitan_area","country","world"]
    df['distance_group'] = pd.cut(df['distance'],right=True, bins=bins, labels=labels, include_lowest=True)

In [120]:


clean_data(df)
add_time(df)
add_age(df)
add_city_group(df)
add_distance(df)
add_distance_bucket(df)
df.to_csv("../data/processed/cleeaned_fraud_train.csv", index=False)

In [121]:
df

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,trans_num,unix_time,merch_lat,merch_long,is_fraud,date_of_birth,week_day,hour,age,age_group,city_group,distance,distance_group
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,1988-03-09,Tuesday,0,31,25-34,small_town,78.597680,metropolitan_area
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,1978-06-21,Tuesday,0,41,35-49,rural,30.212218,metropolitan_area
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,1962-01-19,Tuesday,0,57,50-64,small_town,108.206235,country
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0,1967-01-12,Tuesday,0,52,50-64,small_town,95.673366,metropolitan_area
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0,1986-03-28,Tuesday,0,33,25-34,rural,77.556853,metropolitan_area
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,2020-06-21 12:12:08,30263540414123,fraud_Reichel Inc,entertainment,15.56,Erik,Patterson,M,162 Jessica Row Apt. 072,Hatch,UT,84735,37.7175,-112.4777,258,Geoscientist,440b587732da4dc1a6395aba5fb41669,1371816728,36.841266,-111.690765,0,1961-11-24,Sunday,12,59,50-64,rural,119.752305,country
1296671,2020-06-21 12:12:19,6011149206456997,fraud_Abernathy and Sons,food_dining,51.70,Jeffrey,White,M,8617 Holmes Terrace Suite 651,Tuscarora,MD,21790,39.2667,-77.5101,100,"Production assistant, television",278000d2e0d2277d1de2f890067dcc0a,1371816739,38.906881,-78.246528,0,1979-12-11,Sunday,12,41,35-49,rural,75.104191,metropolitan_area
1296672,2020-06-21 12:12:32,3514865930894695,fraud_Stiedemann Ltd,food_dining,105.93,Christopher,Castaneda,M,1632 Cohen Drive Suite 639,High Rolls Mountain Park,NM,88325,32.9396,-105.8189,899,Naval architect,483f52fe67fabef353d552c1e662974c,1371816752,33.619513,-105.130529,0,1967-08-30,Sunday,12,53,50-64,rural,99.047874,metropolitan_area
1296673,2020-06-21 12:13:36,2720012583106919,"fraud_Reinger, Weissnat and Strosin",food_dining,74.90,Joseph,Murray,M,42933 Ryan Underpass,Manderson,SD,57756,43.3526,-102.5411,1126,Volunteer coordinator,d667cdcbadaaed3da3f4020e83591c83,1371816816,42.788940,-103.241160,0,1980-08-18,Sunday,12,40,35-49,small_town,84.627772,metropolitan_area


In [122]:
df

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,trans_num,unix_time,merch_lat,merch_long,is_fraud,date_of_birth,week_day,hour,age,age_group,city_group,distance,distance_group
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,1988-03-09,Tuesday,0,31,25-34,small_town,78.597680,metropolitan_area
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,1978-06-21,Tuesday,0,41,35-49,rural,30.212218,metropolitan_area
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,1962-01-19,Tuesday,0,57,50-64,small_town,108.206235,country
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0,1967-01-12,Tuesday,0,52,50-64,small_town,95.673366,metropolitan_area
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0,1986-03-28,Tuesday,0,33,25-34,rural,77.556853,metropolitan_area
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,2020-06-21 12:12:08,30263540414123,fraud_Reichel Inc,entertainment,15.56,Erik,Patterson,M,162 Jessica Row Apt. 072,Hatch,UT,84735,37.7175,-112.4777,258,Geoscientist,440b587732da4dc1a6395aba5fb41669,1371816728,36.841266,-111.690765,0,1961-11-24,Sunday,12,59,50-64,rural,119.752305,country
1296671,2020-06-21 12:12:19,6011149206456997,fraud_Abernathy and Sons,food_dining,51.70,Jeffrey,White,M,8617 Holmes Terrace Suite 651,Tuscarora,MD,21790,39.2667,-77.5101,100,"Production assistant, television",278000d2e0d2277d1de2f890067dcc0a,1371816739,38.906881,-78.246528,0,1979-12-11,Sunday,12,41,35-49,rural,75.104191,metropolitan_area
1296672,2020-06-21 12:12:32,3514865930894695,fraud_Stiedemann Ltd,food_dining,105.93,Christopher,Castaneda,M,1632 Cohen Drive Suite 639,High Rolls Mountain Park,NM,88325,32.9396,-105.8189,899,Naval architect,483f52fe67fabef353d552c1e662974c,1371816752,33.619513,-105.130529,0,1967-08-30,Sunday,12,53,50-64,rural,99.047874,metropolitan_area
1296673,2020-06-21 12:13:36,2720012583106919,"fraud_Reinger, Weissnat and Strosin",food_dining,74.90,Joseph,Murray,M,42933 Ryan Underpass,Manderson,SD,57756,43.3526,-102.5411,1126,Volunteer coordinator,d667cdcbadaaed3da3f4020e83591c83,1371816816,42.788940,-103.241160,0,1980-08-18,Sunday,12,40,35-49,small_town,84.627772,metropolitan_area
